In [ ]:
# Copyright 2026 Google LLC
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#     https://www.apache.org/licenses/LICENSE-2.0
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

# Module 00 — Setup & Auth (+ Request/Response Logging)

Welcome to the **Claude on the Agent Platform** series. This module gets you wired up and proves a call works end-to-end.

## The one rule of this repo

> **Every call to Claude routes through Google Cloud's Agent Platform** (formerly Vertex AI) using the **`AnthropicVertex`** client. We **never** use the first-party Anthropic API and there is **no `ANTHROPIC_API_KEY`** anywhere. Authentication is Google Cloud **Application Default Credentials (ADC)**.

This client is built once here and imported by every later module.

## By the end of this module you will have

- A single editable config block (project, location, model) with a placeholder guard.
- The `anthropic[vertex]` + BigQuery dependencies installed.
- ADC verified and a working `AnthropicVertex` client.
- Optional **request/response logging** to BigQuery, enabled and verified.
- A confirmed hello-world response from **`claude-opus-4-8`**.
- The knowledge to pick endpoints (global vs regional) and run Claude Code on the same path.

## Prerequisites

- **A GCP project with billing enabled.**
- **Claude Opus 4.8 enabled in Model Garden** for that project (Model Garden → Anthropic → Claude Opus 4.8 → *Enable*).
- **`gcloud` + ADC available.**
  - In **Cloud Shell** and **Vertex AI Workbench** this is automatic.
  - **Locally**, run once: `gcloud auth application-default login`.

No API keys are required — ADC is the only credential.

## Part B — Config & client

One editable config block. All user-specific values are placeholders with smart defaults — we try to auto-detect your project from `gcloud`.

In [ ]:
# ===== EDITABLE CONFIG (the only block you should need to touch) =====
import subprocess

def _default_project() -> str:
    """Best-effort read of the active project from `gcloud config get-value project`."""
    try:
        out = subprocess.run(
            ["gcloud", "config", "get-value", "project"],
            capture_output=True, text=True, timeout=15,
        ).stdout.strip()
        # gcloud prints "(unset)" when no project is configured.
        if out and out != "(unset)":
            return out
    except Exception:
        pass
    return "<YOUR_PROJECT_ID>"

# Auto-detect the project; falls back to the sentinel if gcloud can't tell us.
PROJECT_ID = _default_project()   # or hard-code: PROJECT_ID = "my-project-id"
LOCATION   = "global"             # "global" endpoint; see Part E for regional.
MODEL      = "claude-opus-4-8"    # exact model string — do not change.

print(f"PROJECT_ID = {PROJECT_ID}")
print(f"LOCATION   = {LOCATION}")
print(f"MODEL      = {MODEL}")

In [ ]:
# Guard: make sure the placeholder was actually replaced.
assert PROJECT_ID and PROJECT_ID != "<YOUR_PROJECT_ID>", (
    "PROJECT_ID is still the placeholder. Set it in the config block above — "
    "either run `gcloud config set project <id>` so it auto-detects, or edit "
    "PROJECT_ID directly."
)
print(f"✅ Project looks good: {PROJECT_ID}")

### Install dependencies

`anthropic[vertex]` provides the `AnthropicVertex` client; `google-cloud-bigquery` is used for the optional logging setup.

In [ ]:
%pip install --quiet "anthropic[vertex]" google-cloud-bigquery

### Verify ADC

Confirm Application Default Credentials resolve and carry the `cloud-platform` scope. If this fails locally, run `gcloud auth application-default login`.

In [ ]:
import google.auth

credentials, adc_project = google.auth.default(
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
print("✅ ADC resolved.")
print(f"   ADC-detected project: {adc_project}")
print(f"   Using PROJECT_ID:     {PROJECT_ID}")

### Build the client

This `client` is the single entry point for every Claude call in the series — later modules import it. Note there is **no API key**: `AnthropicVertex` uses ADC under the hood.

In [ ]:
from anthropic import AnthropicVertex

client = AnthropicVertex(project_id=PROJECT_ID, region=LOCATION)
print(f"✅ AnthropicVertex client ready (project={PROJECT_ID}, region={LOCATION}).")

## Part C — Logging (optional but recommended)

**Why log request/response pairs?**

- **Audit** — a durable record of what was asked and answered.
- **Debug** — inspect exact payloads when behavior surprises you.
- **Cost** — token usage per call for attribution and forecasting.
- **Eval** — a corpus to score, replay, and regression-test against.

**Caveats to know before enabling:**

- Request/response logging is a **Preview** feature and may change.
- Logs land in **BigQuery**, which incurs **storage and query cost** — you own that table.
- For Anthropic models this is configured via the **REST API only** (no console toggle / SDK helper).
- Logging captures **only platform-routed calls** (i.e., calls through `AnthropicVertex`). Anything not routed through the Agent Platform is invisible to it — which is exactly why the one-rule matters.

In [ ]:
# Logging destination config.
DATASET       = "claude_logging"
TABLE         = "claude_opus_4_8_logs"
SAMPLING_RATE = 1.0          # 1.0 = log every call; lower to sample.
BQ_LOCATION   = "US"         # BigQuery dataset location.
BQ_URI        = f"bq://{PROJECT_ID}.{DATASET}.{TABLE}"

print(f"BQ_URI = {BQ_URI}")

### Create the BigQuery dataset (idempotent)

Vertex creates the destination **table** on first logged call; we just need the **dataset** to exist. This is safe to re-run.

In [ ]:
from google.cloud import bigquery

bq = bigquery.Client(project=PROJECT_ID)
dataset_id = f"{PROJECT_ID}.{DATASET}"

try:
    bq.get_dataset(dataset_id)
    print(f"✅ Dataset already exists: {dataset_id}")
except Exception:
    ds = bigquery.Dataset(dataset_id)
    ds.location = BQ_LOCATION
    bq.create_dataset(ds, exists_ok=True)
    print(f"✅ Created dataset: {dataset_id} (location={BQ_LOCATION})")

### Enable logging via REST

We refresh an ADC access token and POST a `setPublisherModelConfig` request for the Anthropic publisher model. The host depends on the endpoint: the `global` endpoint uses `aiplatform.googleapis.com`, regional endpoints are prefixed.

In [ ]:
import requests
import google.auth.transport.requests

# Refresh an ADC access token for the REST call.
auth_req = google.auth.transport.requests.Request()
credentials.refresh(auth_req)
token = credentials.token

HOST = "aiplatform.googleapis.com" if LOCATION == "global" else f"{LOCATION}-aiplatform.googleapis.com"
BASE = (
    f"https://{HOST}/v1beta1/projects/{PROJECT_ID}/locations/{LOCATION}"
    f"/publishers/anthropic/models/{MODEL}"
)

headers = {"Authorization": f"Bearer {token}", "Content-Type": "application/json"}
body = {
    "publisherModelConfig": {
        "loggingConfig": {
            "enabled": True,
            "samplingRate": SAMPLING_RATE,
            "bigqueryDestination": {"outputUri": BQ_URI},
            "enableOtelLogging": True,
        }
    }
}

resp = requests.post(f"{BASE}:setPublisherModelConfig", headers=headers, json=body)
print(f"Status: {resp.status_code}")
print(resp.text)

### Verify logging is enabled

Fetch the current config and assert logging is on.

In [ ]:
verify = requests.get(f"{BASE}:fetchPublisherModelConfig", headers=headers)
print(f"Status: {verify.status_code}")
print(verify.text)

cfg = verify.json().get("loggingConfig", {})
assert cfg.get("enabled") is True, f"Logging is not enabled: {cfg}"
print("✅ loggingConfig.enabled == True")

## Part D — Prove it end-to-end

Make a real call through the `AnthropicVertex` client.

In [ ]:
message = client.messages.create(
    model=MODEL,
    max_tokens=512,
    messages=[{
        "role": "user",
        "content": "In one sentence, say hello and confirm you are running via Google Cloud's Agent Platform.",
    }],
)

print(message.content[0].text)
print("-" * 60)
print(f"stop_reason : {message.stop_reason}")
print(f"input_tokens : {message.usage.input_tokens}")
print(f"output_tokens: {message.usage.output_tokens}")

### Inspect the log table

Query the most recent log rows. **Logging is asynchronous** — the destination table is created after the first logged call and rows can take a minute or two to appear. If this errors or returns nothing, wait 1–2 minutes and re-run.

In [ ]:
query = f"""
SELECT logging_time, model, api_method
FROM `{PROJECT_ID}.{DATASET}.{TABLE}`
ORDER BY logging_time DESC
LIMIT 5
"""

try:
    for row in bq.query(query).result():
        print(row.logging_time, row.model, row.api_method)
except Exception as e:
    print("ℹ️  No rows yet (or table not created).")
    print("   Logging is ASYNC — the table appears after the first call.")
    print("   Wait 1–2 minutes and re-run this cell.")
    print(f"   ({type(e).__name__}: {e})")

## Part E — Two things every user needs

### 1. Endpoint selection (global vs regional)

The **`global`** endpoint (used above) offers the broadest availability and is the recommended default. **Regional** endpoints pin traffic to a specific location for data-residency or latency reasons — build the client by passing that region instead.

> ⚠️ **Confirm the region string on the Claude Opus 4.8 Model Garden card before relying on it.** The example below uses `us-east5` purely for illustration. Docs navigation lags for 4.x models, so do **not** treat any region string here as guaranteed — the Model Garden card for Opus 4.8 is the source of truth for where it's served.

In [ ]:
# Illustrative regional client — region must be CONFIRMED on the Opus 4.8 Model Garden card.
REGIONAL_LOCATION = "us-east5"  # ⚠️ example only; verify availability before use.
regional_client = AnthropicVertex(project_id=PROJECT_ID, region=REGIONAL_LOCATION)
print(f"Built an illustrative regional client for region={REGIONAL_LOCATION!r}.")
print("⚠️  Confirm this region serves claude-opus-4-8 on the Model Garden card before using it.")

### 2. Claude Code on the same path

Claude Code can route through the Agent Platform too — same auth (ADC), same model, no API key. Set these environment variables, then run `claude`:

```bash
export CLAUDE_CODE_USE_VERTEX=1
export CLOUD_ML_REGION=global
export ANTHROPIC_VERTEX_PROJECT_ID=<YOUR_PROJECT_ID>
export ANTHROPIC_MODEL=claude-opus-4-8

claude
```

> ℹ️ Variable names occasionally change — verify the current names against the **Claude Code docs** before relying on them.

## Part F — Cleanup & recap

### Disable logging (commented out on purpose)

Leave this **commented out** so it can't run by accident. Uncomment and run only when you genuinely want to turn logging off.

In [ ]:
# --- DISABLE LOGGING (uncomment to run) ---
# disable_body = {"publisherModelConfig": {"loggingConfig": {"enabled": False}}}
# disable_resp = requests.post(f"{BASE}:setPublisherModelConfig", headers=headers, json=disable_body)
# print(f"Status: {disable_resp.status_code}")
# print(disable_resp.text)

### Recap

- **One rule:** every Claude call routes through the Agent Platform via `AnthropicVertex` — no first-party API key, ADC only.
- Model string is exactly **`claude-opus-4-8`**.
- You have a single config block (with a placeholder guard), a verified `client`, and a confirmed hello-world response.
- Optional **BigQuery logging** is enabled and verified; remember it's **async** and incurs BigQuery cost.
- You know how to target **global vs regional** endpoints (confirm regions on the Model Garden card) and run **Claude Code** on the same path.

**Next: Module 01 — Text generation.**